# 01. 상권 레이더 — 상권×업종의 다음 분기 매출 예측 (LightGBM)

**질문**: 이 상권에서 이 업종의 *다음 분기* 매출은 얼마가 될까? 그리고 같은 업종 평균보다 더 오를까(성장), 떨어질까(하락)?

- 입력 파일: `data/features_ind_v1.parquet` (행 = 상권 × 업종 × 분기), `data/features_ind_v1_spec.json` (피처 목록)
- 한 행의 뜻: "2025년 2분기, 성수동카페거리, 한식음식점" 같은 조합 하나. `y`는 그 분기 매출, `y_next`는 **다음 분기 매출(정답)**.
- 분할은 시간 순서: train(타깃 분기 ≤ 2025Q2) → val(2025Q3~Q4, 조기종료용) → test(2026Q1~Q2). 미래 정보가 학습에 섞이지 않게 하는 것이 핵심.
- 모델은 매출 자체가 아니라 **변화량** `log1p(y_next) − log1p(y)`를 학습하고, 매출이 큰 상권에 조금 더 가중(√매출)을 준다.

In [1]:
import json
import numpy as np
import pandas as pd
import lightgbm as lgb
from pathlib import Path

# 데이터 위치: 이 폴더의 data/ 에 넣었으면 그걸, 아니면 프로젝트의 data/processed/ 를 쓴다
DATA = next(p for p in [Path("data"), Path("../data/processed")] if (p / "features_ind_v1.parquet").exists())
df = pd.read_parquet(DATA / "features_ind_v1.parquet")
spec = json.load(open(DATA / "features_ind_v1_spec.json", encoding="utf-8"))
NUM, CAT = spec["features_num"], spec["features_cat"]          # 수치 피처 91개, 범주 피처 7개

df = df[df["split"] != "forecast"].reset_index(drop=True)      # forecast = 정답(y_next)이 아직 없는 마지막 분기 → 학습·평가에서 제외
print(df.shape, "| 분할:", df["split"].value_counts().to_dict())
print("분기 범위:", df["기준_년분기_코드"].min(), "~", df["기준_년분기_코드"].max(), "| 상권", df["상권_코드"].nunique(), "곳 × 업종", df["서비스_업종_코드"].nunique(), "개")
df[["기준_년분기_코드", "상권_코드", "서비스_업종_코드", "y", "y_next", "split"]].head()

(448187, 114) | 분할: {'train': 365013, 'val': 41774, 'test': 41400}
분기 범위: 20211 ~ 20261 | 상권 1591 곳 × 업종 63 개


,기준_년분기_코드,상권_코드,서비스_업종_코드,y,y_next,split
0,20211,3001491,CS100001,6.414664e+09,8.509195e+09,train
1,20212,3001491,CS100001,8.509195e+09,6.393376e+09,train
2,20213,3001491,CS100001,6.393376e+09,9.899183e+09,train
3,20214,3001491,CS100001,9.899183e+09,7.687103e+09,train
4,20221,3001491,CS100001,7.687103e+09,1.347932e+10,train


## 1. 피처 준비와 시간 분할
피처는 크게 다섯 묶음이다. ① 매출 자기 이력(지난 1~4분기, 전년 대비, 4분기 평균·변동) ② 같은 업종 전체 대비 상대 수준 ③ 점포 수·개업률·폐업률·프랜차이즈 비율 ④ 상권 단위 유동인구·직장인구·상주인구·지하철·집객시설 ⑤ 분기(계절)·상권 유형·자치구 같은 범주형.
LightGBM은 범주형을 `category` dtype으로 주면 알아서 처리한다.

In [2]:
tr, va, te = (df["split"] == s for s in ("train", "val", "test"))

X = df[NUM + CAT].copy()
for c in CAT:                                                   # 범주형: 전체 테이블 기준으로 카테고리 집합 고정 (train/val/test 코드 일치)
    cats = sorted(df[c].dropna().astype(str).unique())
    X[c] = pd.Categorical(X[c].astype(str).where(X[c].notna(), None), categories=cats)

y_delta = np.log1p(df["y_next"]) - df["log_y"]                  # 학습 타깃 = 다음 분기 변화량(로그)
w = np.sqrt(df["y"])                                            # 가중 = √매출 (큰 상권 조금 더 중요)
print("train/val/test:", tr.sum(), va.sum(), te.sum())

train/val/test: 365013 41774 41400


## 2. 기준선 — 모델이 이겨야 하는 상대
아무 모델 없이도 할 수 있는 예측 3개. **이걸 못 이기면 모델을 쓸 이유가 없다.**

In [3]:
def mae_log(y, p):  return float(np.mean(np.abs(np.log1p(y) - np.log1p(p))))     # 로그 오차 평균 (비율 오차에 가까움)
def wape(y, p):     return float(np.abs(y - p).sum() / y.sum())                      # 매출 가중 절대 오차율
def mdape(y, p):    return float(np.median(np.abs(y - p) / y))                       # 오차율의 중앙값

preds = {
    "지난 분기 값":        df["y"],
    "최근 4분기 평균":     df["roll_mean4_raw"],
    "작년 같은 분기":      df["y_lag3"],
}

## 3. LightGBM 학습
val에서 성능이 더 안 좋아지면 멈춘다(조기종료). 파라미터는 프로젝트에서 쓴 값 그대로.

In [4]:
PARAMS = dict(objective="regression", metric="l1", learning_rate=0.1, num_leaves=63, min_data_in_leaf=200,
              feature_fraction=0.7, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0, max_bin=127, verbose=-1, seed=42, num_threads=8)

dtr = lgb.Dataset(X[tr], y_delta[tr], weight=w[tr])
dva = lgb.Dataset(X[va], y_delta[va], weight=w[va])
model = lgb.train(PARAMS, dtr, num_boost_round=3000, valid_sets=[dva],
                  callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
print("멈춘 라운드:", model.best_iteration)

pred_delta = model.predict(X, num_iteration=model.best_iteration)
preds["LightGBM (변화량 + √매출 가중)"] = pd.Series(np.expm1(pred_delta + df["log_y"]), index=df.index)   # 변화량 → 매출로 되돌림

멈춘 라운드: 124


## 4. 결과 비교 (test = 2026Q1~Q2)
세 지표 모두 작을수록 좋다.

In [5]:
ok = pd.concat(preds.values(), axis=1).notna().all(axis=1)      # 이력이 짧아 기준선을 못 만드는 행은 빼고, 모든 모델을 같은 행에서 비교
rows = []
for name, p in preds.items():
    for sp, m in (("val", va & ok), ("test", te & ok)):
        rows.append({"모델": name, "구간": sp, "MAE_log": mae_log(df.loc[m, "y_next"], p[m]), "WAPE": wape(df.loc[m, "y_next"], p[m]), "MdAPE": mdape(df.loc[m, "y_next"], p[m])})
print("비교 행 수: val", int((va & ok).sum()), "/ test", int((te & ok).sum()))
res = pd.DataFrame(rows).pivot(index="모델", columns="구간", values=["MAE_log", "WAPE", "MdAPE"]).round(4)
res

비교 행 수: val 39865 / test 39488


MAE_log            WAPE           MdAPE        
구간                         test     val    test     val    test     val
모델                                                                     
LightGBM (변화량 + √매출 가중)  0.2358  0.2475  0.1051  0.1222  0.1134  0.1266
작년 같은 분기                 0.3515  0.3496  0.1543  0.1639  0.1705  0.1761
지난 분기 값                  0.2734  0.2728  0.1348  0.1397  0.1372  0.1368
최근 4분기 평균                0.2635  0.2686  0.1207  0.1394  0.1350  0.1384

## 5. 성장 / 유지 / 하락 판정
예측 매출을 작년 같은 분기와 비교한 성장률에서 **같은 업종의 중앙값을 뺀 값**이 +15%p 이상이면 성장, −15%p 이하면 하락. 업종 전체가 오르는 계절 효과를 빼고 "그 업종 안에서 상대적으로" 보려는 것.
정답 라벨 `cls_next`도 같은 규칙으로 만들어져 있다. 기준선은 "지금 분기의 판정이 다음 분기에도 그대로"(persistence).

In [6]:
from sklearn.metrics import accuracy_score, f1_score

def to_class(pred_sales):
    g = np.log1p(pred_sales) - df["log_y_lag3"]                                      # 작년 동기 대비 성장률(로그)
    g = g - g.groupby([df["서비스_업종_코드"], df["q_next"]]).transform("median")     # 같은 업종·같은 분기 중앙값 제거
    return pd.Series(np.where(g > 0.15, "성장", np.where(g < -0.15, "하락", "유지")), index=df.index)

cls = {"persistence (현재 판정 유지)": df["cls_cur"].fillna("유지"),          # 이력이 짧아 판정이 없는 행은 '유지'로
       "최근 4분기 평균 → 판정": to_class(df["roll_mean4_raw"]), "LightGBM → 판정": to_class(preds["LightGBM (변화량 + √매출 가중)"])}
ok = te & df["cls_next"].notna()
pd.DataFrame({name: {"Accuracy": accuracy_score(df.loc[ok, "cls_next"], p[ok]), "Macro-F1": f1_score(df.loc[ok, "cls_next"], p[ok], average="macro")} for name, p in cls.items()}).T.round(3)

,Accuracy,Macro-F1
persistence (현재 판정 유지),0.604,0.589
최근 4분기 평균 → 판정,0.665,0.636
LightGBM → 판정,0.694,0.683


## 6. 어떤 피처가 중요했나 (gain 기준 상위 15)

In [7]:
imp = pd.Series(model.feature_importance("gain"), index=model.feature_name())
(imp / imp.sum() * 100).sort_values(ascending=False).head(15).round(2).to_frame("중요도 %")

,중요도 %
dev_from_mean4,20.54
서비스_업종_코드,15.43
season_next,9.64
qoq_t,7.22
자치구_코드,7.08
roll_std4,3.16
log_ticket,2.89
a_spend,1.67
yoy_t_lag1,1.56
log_y,1.39


## 읽는 법
- 기대값(프로젝트 기록, test): LightGBM이 "최근 4분기 평균"보다 MAE_log 기준 약 10% 낮고, 판정 Accuracy는 persistence 0.61 → 약 0.69.
- 모델이 못 이기는 업종도 있다. 전체 평균만 보지 말고 업종별로 나눠 보면 좋다.
- **하지 말 것**: 무작위 분할(KFold). 같은 상권의 미래 분기가 학습에 들어가 점수가 부풀려진다.